## Requirements
- A pre-created vector search endpoint
- Serverless compute v4
- Note: using "skinny" version of mlflow does not show the complete tool call tree in the tracing UI. Use the full lib instead.

In [0]:
%pip install "databricks-vectorsearch==0.60" "mlflow[databricks]==3.6.0" "databricks-langchain==0.9.0" "langchain==1.0" -q
%restart_python


In [0]:
catalog = "workspace"
schema = "feature_model"
index = "docs_chunked_index"
model_name = "imda_knowledge_assistant_with_resources"
tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "imda_knowledge_base"
}

## A. Testing Vector Search in AI Playground
- Objective: quickly check if the vector search index is working correctly and understand how the retrieval system responds to different queries.
- Concept: the vector search index is treated as a tool to the main agent LLM.

## B. Builing a RAG agent with LangChain
- use "agent as code" approach: write agent implementation to a python file (`agent.py`) - the recommended method when logging the models with mlflow.
- the agent will use the UC's vector search as a tool.

### B1. Enable MLflow Tracing
Note: MLflow tracing is enabled by default in classic compute. On serverless compute, this has to be done manually.

In [0]:
import mlflow

mlflow.langchain.autolog()

### B2. Create the Agent

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel
from typing import Callable, Optional



def build_agent(
    llm_endpoint: str, index_name: str, num_results: int = 3, system_prompt: str = ""
):
    # init the model with OpenAI standard I/O schemas
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=500)

    vs_tool = VectorSearchRetrieverTool(
        name="imda_llm_testing_knowledge_search",
        index_name=index_name,
        description="Search the IMDA's document `Starter Kit for Testing LLM-Based Applications for Safety and Reliability` for relevant information on testing LLM-based applications.",
        num_results=num_results,
    )
    tools = [vs_tool]

    # Optional: use an in-memory saver to save the agent's state
    checkpointer = InMemorySaver()

    agent = create_agent(
        model=model, tools=tools, system_prompt=system_prompt, checkpointer=checkpointer
    )
    return agent

# to be loaded from config file:
LLM_ENDPOINT_NAME = "databricks-qwen35-122b-a10b"
SYSTEM_PROMPT = """
    You are the knowledge assistant for Singapore's Infocomm Media Development Authority (IMDA), specialized on  testing LLM-based applications for safety and reliability. Respond in a clear, professional, and factual tone appropriate for developers. Use only verified information from the internal documents, and include source references when available. If the answer cannot be found, clearly state that, and suggest related sections or next steps. Do not speculate, make assumptions, or provide informaiton outside of the provided context.
"""
INDEX_NAME = f"{catalog}.{schema}.{index}"
NUM_RESULTS = 3

# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "databricks-demo-1"}}

# init an agent
agent = build_agent(
    llm_endpoint=LLM_ENDPOINT_NAME, system_prompt=SYSTEM_PROMPT, index_name=INDEX_NAME,
    num_results=NUM_RESULTS
)

In [0]:
# quick smoke test
user_query = "What is the purpose of this IMDA LLM testing starter kit?"

response = agent.invoke(
    {"messages":[{"role": "user", "content": user_query}]},
    config=config
)
print(response['messages'][-1].content)

## C. Log the Agent to Model Registry
- Create the model (agent) from code script.
- Create a .yaml config file that contains important configs like LLM endpoint, index name... to be logged along the agent.

### C1. Write agent config to yaml

In [0]:
import yaml


# helper function
def create_config(
    llm_endpoint: str, index_name: str, num_results: int = 3, system_prompt: str = ""
):
    """
    Create a minimal YAML config for the agent.
    """
    config = {
        "llm_endpoint": llm_endpoint,
        "vector_search": {"index_name": index_name, "num_results": num_results},
        "system_prompt": system_prompt,
    }
    return config


# create config file
agent_config = create_config(
    llm_endpoint=LLM_ENDPOINT_NAME,
    system_prompt=SYSTEM_PROMPT,
    index_name=INDEX_NAME,
    num_results=NUM_RESULTS,
)

with open("agent-config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(agent_config, f, sort_keys=False)

print("Config file written: agent-config.yaml")
print(yaml.safe_dump(agent_config, sort_keys=False))

### C2. Write agent code to a file

In [0]:
%%writefile agent.py
import os
from uuid import uuid4
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langgraph.checkpoint.memory import InMemorySaver


# load agent config from yaml file (in the same directory)
def _load_config(path: str = "agent-config.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at: {path}")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint")
    vs = cfg.get("vector_search", {}) or {}
    index_name = vs.get("index_name")
    num_results = int(vs.get("num_results", 3))
    system_prompt = cfg.get("system_prompt")
    return {
        "llm_endpoint": llm_endpoint,
        "index_name": index_name,
        "num_results": num_results,
        "system_prompt": system_prompt
    }


# build LangChain agent with the config above:
# this is the same code as the smoke test above
def build_agent(
    llm_endpoint: str, index_name: str, num_results: int = 3, system_prompt: str = ""
):
    # init the model with OpenAI standard I/O schemas
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=500)

    vs_tool = VectorSearchRetrieverTool(
        name="imda_llm_testing_knowledge_search",
        index_name=index_name,
        description="Search the IMDA's document `Starter Kit for Testing LLM-Based Applications for Safety and Reliability` for relevant information on testing LLM-based applications.",
        num_results=num_results,
    )
    tools = [vs_tool]

    # Optional: use an in-memory saver to save the agent's state
    checkpointer = InMemorySaver()

    agent = create_agent(
        model=model, tools=tools, system_prompt=system_prompt, checkpointer=checkpointer
    )
    return agent

# MLflow ResponsesAgent interface implementation for LangChain agent
class LangChainResponsesAgent(ResponsesAgent):
    def __init__(self):
        cfg = _load_config()
        self._cfg = cfg
        self._agent = build_agent(
            llm_endpoint=cfg["llm_endpoint"],
            index_name=cfg["index_name"],
            num_results=cfg["num_results"]
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        msgs = [m.model_dump() for m in request.input] # example: [{'role': 'user' | 'assistant', 'content': '...'}, ...]
        # _ = _last_user_text(msgs) if msgs else ""

        # Generate a unique thread ID for each pred:
        thread_id = f"imda-{uuid4()}"
        result = self._agent.invoke(
            {"messages": msgs},
            config={"configurable": {"thread_id": thread_id}},
        )

        # Extract agent response text
        try:
            text = result["messages"][-1].content
        except Exception:
            text = str(result)
        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text, str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )

# get the model obj for mlflow:
AGENT = LangChainResponsesAgent()
mlflow.models.set_model(AGENT)

## D. Logging and Registering Agent as a Model
- Logging captures the agent code, config, dependencies, and metadata in structured format that can be versioned, tracked and deployed.
- When logging a model, we create a run that records:
    - model artifacts: the agent code (`agent.py`) and config (`agent-config.yaml`)
    - dependencies: python packages required to run the agent
    - resources: external dependencies like vector search index and serving endpoint
    - I/O examples: sample data that demonstrates the expected model interface
    - metadata and tags: info about the model version, purpose and lineage
- the logged model becomes a reproducible artifact that can be loaded, tested and deployed in any environment with the same dependencies and resources available.

### D1. Logging

In [0]:
import mlflow
from importlib.metadata import version as get_version
from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksVectorSearchIndex,
)


# create an input example for the model signature:
input_example = {
    "input": [
        {"role": "user", "content": "What is the purpose of this IMDA LLM testing starter kit?"}
    ]
}

# Define the resources your model needs for inference
resources=[
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
    DatabricksVectorSearchIndex(index_name=INDEX_NAME),
]

# start an MLflow run and log the model
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent.py", # path to model from code
        code_paths=["agent-config.yaml"], # path to extra files
        input_example=input_example,
        pip_requirements=[
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"langchain=={get_version('langchain')}",
            f"mlflow=={get_version('mlflow')}",
        ],
        resources=resources
    )
    model_uri = logged_agent_info.model_uri

print(f"Model logged sucessfully with URI: {model_uri}")

### D2. Reloading and Validation of I/O

In [0]:
output_example = mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/{model_name}",
    input_data=[input_example],
    env_manager="uv", # mlflow recommends using uv for better performance
)
output_example

In [0]:
# another method:
# Load the model from the model URI:
pyfunc_model = mlflow.pyfunc.load_model(model_uri=model_uri)

# Use the input example that was logged with the model:
input_data = pyfunc_model.input_example
print("Input data:")
print(input_data)
print("\n" + "="*50 + "\n")

# Make preduction using the loaded model
# note: the model is reloaded via URI again
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

print("Agent response:")
print(result) # This is None while response is generated from the predict() method, also seen in online training, TBC

### D3. Register the model to UC
Logging and registering serve distinct purposes in the MLOps lifecycle.
- Logging: creates a versioned artifact within an MLflow experiment run. It captures the model code, dependencies and metadata at a specific point in time. Logged models are tied to individial runs and are primarily used for experimentation and development.
- Registering: promotes a logged model to the Model Registry, making it a managed, governed asset with a unique name in the UC. Registered models support:
    - Version management
    - Aliases
    - Governance: permissioins, tags, lineage tracking.
    - Deployment: serve models directly from registry to production endpoints.
- By registering, we transform the agent from an experimental artifact into a production-ready asset that can be discovered, govern, and deployed across the organization.

In [0]:
# Set the registry URI to UC:
mlflow.set_registry_uri("databricks-uc")

# Define the fully qualified model name in the UC
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# Register the model
uc_registered_model_info = mlflow.register_model(
    model_uri=model_uri,
    name=UC_MODEL_NAME
)

print(f"Model registered successfully to Unity Catalog.")
print(f"Model name: {UC_MODEL_NAME}")
print(f"Model version: {uc_registered_model_info.version}")